# Resume / Candidate Screening System Pipeline
This notebook covers the machine learning pipeline to build a resume screening system.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

## 1. Load dataset

In [ ]:
df = pd.read_csv('../data/resumes.csv')
df.head()

## 2. Explore dataset

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

plt.figure(figsize=(10,6))
sns.countplot(y='job_role', data=df, order=df['job_role'].value_counts().index)
plt.title('Distribution of Job Roles')
plt.show()

## 3. Text preprocessing

In [ ]:
df.dropna(subset=['resume_text', 'skills', 'job_role'], inplace=True)
df['combined_text'] = df['resume_text'] + " " + df['skills']

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

df['cleaned_text'] = df['combined_text'].apply(clean_text)
df[['job_role', 'cleaned_text']].head()

## 4. TF-IDF feature extraction

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['job_role']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

## 5. Train Logistic Regression and Naive Bayes

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

## 6. Compare accuracy

In [ ]:
lr_acc = accuracy_score(y_test, lr_pred)
nb_acc = accuracy_score(y_test, nb_pred)

print(f"Logistic Regression Accuracy: {lr_acc * 100:.2f}%")
print(f"Multinomial Naive Bayes Accuracy: {nb_acc * 100:.2f}%")

if lr_acc >= nb_acc:
    best_model = lr_model
    best_name = "Logistic Regression"
    best_pred = lr_pred
else:
    best_model = nb_model
    best_name = "Multinomial Naive Bayes"
    best_pred = nb_pred

print(f"\nBest Model: {best_name}")

## 7. Classification report

In [ ]:
print(f"Classification Report for {best_name}:\n")
print(classification_report(y_test, best_pred))

## 8. Test with sample resumes

In [ ]:
sample_resumes = [
    "I have 3 years of experience as a Data Analyst using SQL, Tableau, and Python. I enjoy cleaning data and building dashboards.",
    "Passionate about web development. I am a Frontend Developer skilled in React, JavaScript, HTML, and CSS.",
    "Experienced ML Engineer. I build predictive models using TensorFlow, PyTorch, and Scikit-Learn."
]

for i, resume in enumerate(sample_resumes, 1):
    cleaned = clean_text(resume)
    vectorized = vectorizer.transform([cleaned])
    prediction = best_model.predict(vectorized)[0]
    
    print(f"Sample {i}: {resume}")
    print(f"Predicted Role: {prediction}")
    print("-" * 50)

## 9. Save model and vectorizer

In [ ]:
os.makedirs('../models', exist_ok=True)

with open('../models/resume_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
    
with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully to the models/ directory!")